# Github

https://github.com/WilliamKA02/CSS_Assignments

# Contributions

All members (William, Tobias & Laura) contributed equally to the assignment.

# Imports

In [4]:
import requests
import pandas as pd
import time
import os
from difflib import SequenceMatcher
from joblib import Parallel, delayed
from tqdm import tqdm

# Formalia

Please read the [assignment overview page](https://laura.alessandretti.com/comsocsci2026/wiki_pages/Assignments.html) carefully before proceeding. The page contains information about formatting (including formats etc), group sizes, and many other aspects of handing in the assignment. 

__If you fail to follow these simple instructions, it will negatively impact your grade!__

**Due date and time**: The assignment is due on Mar 3rd at 23:59. Hand in your Jupyter notebook file (with extension `.ipynb`) via DTU Learn _(Assignment 1)_. 

Remember to include in the first cell of your notebook:
* the link to your group's Git repository 
* group members' contributions


## Part 1: Ready Made vs Custom Made Data

> **Exercise: Ready made data vs Custom made data** In this exercise, I want to make sure you have understood they key points of my lecture and the reading. 
>
> 1. What are pros and cons of the custom-made data used in Centola's experiment (the first study presented in the lecture) and the ready-made data used in Nicolaides's study (the second study presented in the lecture)? You can support your arguments based on the content of the lecture and the information you read in Chapter 2.3 of the book __(answer in max 150 words)__.


Centola’s randomized experiment supports a clean causal claim: differences in adoption can be attributed to network structure, not pre-existing similarity. Trade-offs are response bias (people may act differently when they know they’re studied), a narrow/low-stakes behavior, a small sample, and an artificial setup that may not generalize.

Nicolaides’s ready-made observational data avoids response bias and captures real-world behavior at massive scale. But it can’t nail causality: patterns could be social influence, homophily, or shared context (weather, local events), and platform features can introduce algorithmic confounding. Controls help, scale boosts power and coverage, but it doesn’t fix identification—and there are also consent/privacy ethics concerns.

> 2. How do you think these differences can influence the interpretation of the results in each study? __(answer in max 150 words)__

Centola’s randomized experiment supports a clean causal claim: differences in adoption can be attributed to network structure, not pre-existing similarity. The trade-off is generalizability—it's a controlled, artificial setting and a narrow, low-stakes behavior, so people may doubt it carries over to real networks.

Nicolaides’s real-world observational data is more natural and broadly applicable, but harder to interpret. Similar running patterns could be social influence, homophily, or shared context (weather, local events). Controls and scale help, yet the identification problem remains, and platform effects can amplify bias.

## Part 2: Find Researchers using the OpenAlex API

> **Exercise 3: Find potential Computational Social Scientists** In this exercise, we'll use the OpenAlex API to compile a list of researchers in the field of Computational Social Science, focusing on those who have attended the IC2S2 conference in 2025. This will not only later on help you understand the landscape of Computational Social Science research but also develop practical skills in data collection and analysis. 
>
> Please read the text of the whole exercise before starting to work on it. 
>
> **Steps**
> 
> 1. **Retreive data.** Consider the set of unique researcher names that you collected in Week 1, Exercise 3. Use the _authors_ endpoint of the [OpenAlex API](https://docs.openalex.org/api-entities/authors) to _search_ these researchers in the database based on their names. Loop through the list and, for each researcher in your list, find: 
>     - their _id_: The OpenAlex ID for this author.
>     - their _display\_name_: The name of the author as a single string.
>     - their _works\_api\_url_: A URL that will get you a list of all this author's works.
>     - their _h\_index_ : The h-index for this author.
>     - their _works\_count_: The number of  Works this this author has created.
>     - their _country\_code_: The country code of their last known institution
> 2. **Data Storage** Store this information in a Pandas DataFrame and save it to file.
>


In [ ]:
# Load researcher names collected in Week 1
names = pd.read_csv("researchers.csv")["name"].tolist()

# Insert Api key from .env file
API_KEY = os.getenv("OPENALEX_API_KEY")

results = []

for i, name in enumerate(names):
    r = requests.get(
        "https://api.openalex.org/authors",
        params={"search": name, "per-page": 1, "api_key": API_KEY}
    )
    data = r.json().get("results", [])

    if data:
        a = data[0]
        # last_known_institutions is a list; take first entry
        institutions = a.get("last_known_institutions") or [{}]
        results.append({
            "search_name":   name,
            "id":            a.get("id"),
            "display_name":  a.get("display_name"),
            "works_api_url": a.get("works_api_url"),
            "h_index":       a.get("summary_stats", {}).get("h_index"),
            "works_count":   a.get("works_count"),
            "country_code":  institutions[0].get("country_code"),
        })
    else:
        results.append({"search_name": name})

    if i % 100 == 0:
        print(f"{i}/{len(names)} done...")

    time.sleep(0.05)  # stay under rate limit

# Save to file D1
df_d1 = pd.DataFrame(results)
df_d1.to_csv("D1.csv", index=False)
print(f"{df_d1['id'].notna().sum()}/{len(df_d1)} researchers found in OpenAlex")
df_d1.head()

In [5]:
import re
import unicodedata
import pandas as pd

# Cyrillic → Latin transliteration (for names stored in Cyrillic script)
CYRILLIC = str.maketrans({
    'а':'a','б':'b','в':'v','г':'g','д':'d','е':'e','ё':'yo','ж':'zh','з':'z',
    'и':'i','й':'y','к':'k','л':'l','м':'m','н':'n','о':'o','п':'p','р':'r',
    'с':'s','т':'t','у':'u','ф':'f','х':'kh','ц':'ts','ч':'ch','ш':'sh',
    'щ':'shch','ъ':'','ы':'y','ь':'','э':'e','ю':'yu','я':'ya',
    'А':'A','Б':'B','В':'V','Г':'G','Д':'D','Е':'E','Ё':'Yo','Ж':'Zh','З':'Z',
    'И':'I','Й':'Y','К':'K','Л':'L','М':'M','Н':'N','О':'O','П':'P','Р':'R',
    'С':'S','Т':'T','У':'U','Ф':'F','Х':'Kh','Ц':'Ts','Ч':'Ch','Ш':'Sh',
    'Щ':'Shch','Ъ':'','Ы':'Y','Ь':'','Э':'E','Ю':'Yu','Я':'Ya',
})

def normalize(name):
    if pd.isna(name) or str(name).strip() == "": return None
    name = str(name)
    name = name.replace("ß", "ss").replace("ẞ", "SS")          # ß → ss
    name = re.sub(r"[''ʻʼ`´\u2018\u2019]", "'", name)          # normalize apostrophes
    name = re.sub(r"[\u2010-\u2015\u2212]", "-", name)          # normalize hyphens
    normed = unicodedata.normalize("NFD", name).encode("ascii", "ignore").decode("ascii").strip()
    if not normed:                                               # fallback for Cyrillic etc.
        normed = unicodedata.normalize("NFD", name.translate(CYRILLIC)).encode("ascii", "ignore").decode("ascii").strip()
    if not normed: return None
    normed = re.sub(r"[^a-zA-Z\s\-']", "", normed).strip()      # strip superscripts, dots, etc.
    return normed.lower() if normed else None

def names_match(search, display):
    sn, dn = normalize(search), normalize(display)
    if not sn or not dn: return False
    s_last = set(re.split(r"[\s\-]+", sn.split()[-1])) - {""}  # parts of search's last word
    d_last = set(re.split(r"[\s\-]+", dn.split()[-1])) - {""}  # parts of display's last word
    d_tokens = set(re.split(r"[\s\-]+", dn)) - {""}            # all tokens in display name
    s_tokens = set(re.split(r"[\s\-]+", sn)) - {""}            # all tokens in search name
    # Primary:   search last name found anywhere in display (handles hyphenated/extended names)
    # Secondary: display last name found anywhere in search (handles reversed name order)
    return bool(s_last & d_tokens) or bool(d_last & s_tokens)

df = pd.read_csv("D1.csv")
df = df[df["display_name"].notna()]
mask = df.apply(lambda r: names_match(r["search_name"], r["display_name"]), axis=1)

df_removed = df[~mask].reset_index(drop=True)
df = df[mask].reset_index(drop=True)

print(f"Rows after cleaning: {len(df)}")
print(f"Rows removed: {len(df_removed)}")
df.to_csv("D1_clean.csv", index=False)

Rows after cleaning: 1349
Rows removed: 139


>    
> **Handling Challenges**
> 
> I expect that, while working on the steps above, you will encounter several obstacles. As you complete this exercise, you are expected to:     
>
>    - Identify problems that arise.      
>    - Improve your solutions to address such problems, making reasonable decisions when data is incomplete or ambiguous.       
>
> **Reflection Questions**
>  Answer the following questions __(max 150 words for each question)__: 
>
>    - Which challenges did you encounter? How did you address them?

The main issue was that after OpenAlex started requiring an API key (with a token budget), we ran out of tokens pretty quickly because our author list is long. We didn’t notice at first, but it turned out the last part of the dataset was missing. We fixed that by splitting the collection across multiple days and then concatenating the partial outputs into one file (D1). On top of that, doing one request at a time is slow since we also had to sleep between calls to stay under the rate limit.

A second issue was matching: we currently take the first search result, but that isn’t always the right author. We looked into checking the top 5 results and picking the best match, but that would cost 5× more tokens and time. Instead, we went with a simple pruning step afterwards, which removes a lot of wrong matches.

>    - Choose one problem you faced while collecting the data and describe your solution. Why did you choose this approach, and what impact might it have on your data? 

The main problem was entity matching where we map each name from our scraped CSV to the correct OpenAlex author. OpenAlex search returns multiple plausible candidates, and as we discussed in the question above we had to deal with rate/time limits, so we couldn’t manually review matches or fetch many candidates per name.

Our solution was a two-step, pragmatic approach:

1. Take only the top search hit (`per-page=1`) to keep requests low and stay within API constraints.
2. Prune obvious mismatches by filtering to cases where the last name in our `search_name` matches the last name in OpenAlex’s `display_name`

We chose this because it’s simple, fast, and dramatically reduces false matches without extra API calls. The impact is a clear precision–recall tradeoff: we likely keep a cleaner set of authors (higher precision), but we also drop some correct people when the naming formats don’t line up (hyphenated surnames, multi-part last names, middle names/initials, or display-name differences), and we still risk keeping a few wrong matches when two different researchers share the same last name. We manually went through the removed names and added rules to keep wrongly pruned names (like cyrillic names, etc.).


## Part 3: Collect Research Articles

> **Exercise 1: Collecting Research Articles from IC2S2 Authors**
>
>In this exercise, we'll leverage the OpenAlex API to gather information on research articles authored by participants of the IC2S2 2025 conference, referred to as *IC2S2 authors*. **Before you start, please ensure you read through the entire exercise.**
>
> 
> **Steps:**
>  
> 1. **Retrieve Data:** Start with the dataset of *IC2S2 authors* you collected in Week 2, Exercise 3 (called dataset D1 in the figure above). Use the OpenAlex API [works endpoint](https://docs.openalex.org/api-entities/works) to fetch their research articles. For each article, retrieve the following details:
>    - _id_: The unique OpenAlex ID for the work.
>    - _publication_year_: The year the work was published.
>    - _cited_by_count_: The number of times the work has been cited by other works.
>    - _author_ids_: The OpenAlex IDs for the authors of the work.
>    - _title_: The title of the work.
>    - _abstract_inverted_index_: The abstract of the work, formatted as an inverted index.
> 
>     **Important Note on Paging:** By default, the OpenAlex API limits responses to 25 works per request. For more efficient data retrieval, I suggest to adjust this limit to 200 works per request. Even with this adjustment, you will need to implement pagination to access all available works for a given query. This ensures you can systematically retrieve the complete set of works beyond the initial 200. Find guidance on implementing pagination [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/paging#cursor-paging).
>
> 2. **Data Storage:** Organize the retrieved information into two Pandas DataFrames and save them to two files in a suitable format:
>    - Dataset D2: The *IC2S2 papers* dataset should include: *id, publication\_year, cited\_by\_count, author\_ids*.
>    - Dataset D3: The *IC2S2 abstracts* dataset should include: *id, title, abstract\_inverted\_index*.
>  
>
> **Filters:**
> To ensure the data we collect is relevant and manageable, apply the following filters:
>     
>    - Only include *IC2S2 authors* with a total work count between 5 and 5,000.    
>    - Retrieve only works that have received more than 10 citations.    
>    - Limit to works authored by fewer than 10 individuals.    
>    - Include only works relevant to Computational Social Science (focusing on: Sociology OR Psychology OR Economics OR Political Science) AND intersecting with a quantitative discipline (Mathematics OR Physics OR Computer Science), as defined by their [Concepts](https://docs.openalex.org/api-entities/works/work-object#concepts). *Note*: here we only consider Concepts at *level=0* (the most coarse definition of concepts).     
>
> **Efficiency Tips:**
> Writing efficient code in this exercise is **crucial**. To speed up your process:
> 
> - **Apply filters directly in your request:** When possible, use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) of the *works* endpoint to apply the filters above directly in your API request, ensuring only relevant data is returned. Learn about combining multiple filters [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists).  
> - **Bulk requests:** Instead of sending one request for each author, you can use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) to query works by multiple authors in a single request. *Note: My testing suggests that can only include up to 25 authors per request.*
> - **Use multiprocessing:** Implement multiprocessing to handle multiple requests simultaneously. I highly recommmend [Joblib’s Parallel](https://joblib.readthedocs.io/en/stable/) function for that, and [tqdm](https://tqdm.github.io/) can help monitor progress of your jobs. Remember to stay within [the rate limit](https://docs.openalex.org/how-to-use-the-api/rate-limits-and-authentication) of 100 requests per second.
>
>
>   
> For reference, employing these strategies allowed me to fetch the data in about 30 seconds using 5 cores on my laptop. I obtained a dataset of approximately 25 MB (including both the *IC2S2 abstracts* and *IC2S2 papers* files).
> 

In [6]:
API_KEY = os.getenv("OPENALEX_API_KEY")

# Load IC2S2 authors (D1) and filter by work count (5–5000)
df_authors = pd.read_csv("D1_clean.csv")
df_authors = df_authors[df_authors["works_count"].between(5, 5000)]
author_ids = df_authors["id"].dropna().str.replace("https://openalex.org/", "").tolist()
print(f"Querying works for {len(author_ids)} authors")

# Define filters
# OpenAlex concept IDs for social science disciplines (Sociology|Psychology|Economics|Political Science)
SOCIAL = "C144024400|C15744967|C162324750|C17744445"
# OpenAlex concept IDs for quantitative disciplines (Mathematics|Physics|Computer Science)
QUANT  = "C33923547|C121332964|C41008148"

# Comma = AND in OpenAlex filter syntax; | inside a field = OR
# (social_science_concept) AND (quant_concept) AND citations>10 AND authors<10
FILTERS = ",".join([
    f"concepts.id:{SOCIAL}",
    f"concepts.id:{QUANT}",
    "cited_by_count:>10",
    "authors_count:<10",
])

FIELDS = "id,publication_year,cited_by_count,authorships,title,abstract_inverted_index"

# Function to fetch all works for a batch of up to 25 authors (with cursor paging)
def fetch_batch(batch_ids):
    author_filter = "|".join(batch_ids)
    papers, abstracts = [], []
    cursor = "*"

    while cursor:
        r = requests.get(
            "https://api.openalex.org/works",
            params={
                "filter":   f"author.id:{author_filter},{FILTERS}",
                "per-page": 200,
                "cursor":   cursor,
                "select":   FIELDS,
                "api_key":  API_KEY,
            }
        )
        if r.status_code != 200:
            break

        data = r.json()
        results = data.get("results", [])

        for w in results:
            # Extract clean author IDs from authorships
            author_ids_list = [
                a["author"]["id"].replace("https://openalex.org/", "")
                for a in w.get("authorships", [])
                if a.get("author") and a["author"].get("id")
            ]
            papers.append({
                "id":               w.get("id"),
                "publication_year": w.get("publication_year"),
                "cited_by_count":   w.get("cited_by_count"),
                "author_ids":       author_ids_list,
            })
            abstracts.append({
                "id":                      w.get("id"),
                "title":                   w.get("title"),
                "abstract_inverted_index": w.get("abstract_inverted_index"),
            })

        # Advance cursor for next page
        cursor = data.get("meta", {}).get("next_cursor")
        time.sleep(0.01)  # stay under 100 req/s

    return papers, abstracts

# Split into batches of 25 and run with parallel threading
batches = [author_ids[i:i+25] for i in range(0, len(author_ids), 25)]

all_results = Parallel(n_jobs=5, backend="threading")(
    delayed(fetch_batch)(batch) for batch in tqdm(batches)
)

# Flatten and deduplicate (same paper may appear for multiple authors)
all_papers    = [p for papers, _  in all_results for p in papers]
all_abstracts = [a for _, abstracts in all_results for a in abstracts]

df_papers    = pd.DataFrame(all_papers).drop_duplicates(subset="id")
df_abstracts = pd.DataFrame(all_abstracts).drop_duplicates(subset="id")

print(f"{len(df_papers)} unique papers (D2)")
print(f"{df_papers['author_ids'].explode().nunique()} unique co-authors in D2")

# Save
df_papers.to_csv("D2.csv", index=False)
df_abstracts.to_csv("D3.csv", index=False)

Querying works for 1193 authors


100%|██████████| 48/48 [00:20<00:00,  2.36it/s]


13324 unique papers (D2)
20836 unique co-authors in D2


>
> **Data Overview and Reflection questions:** Answer the following questions __(max 150 words for each question)__:     
> - **Dataset summary.** How many works are listed in your Dataset D2 (*IC2S2 papers*) dataframe? How many unique researchers have co-authored these works?     

After cleaning the dataset for wrongly matched researchers in D1.csv and then using D1_clean.csv to run this we found that it further filtered out $1349-1193=156$ authors based on our filters. From these 1193 authors we found 13324 unique papters and 20836 unique co-authors have authored these works.

> - **Efficiency in code.** Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?  

First, we batched authors into groups of 25 using OpenAlex's | (OR) filter syntax, so one request fetches works for 25 authors at once instead of 25 separate calls. Second, we parallelised the batches using joblib's threading backend with 5 workers, so multiple batches run concurrently.

We also used cursor-based pagination (cursor=*) rather than offset pagination, which is what OpenAlex recommends for large result sets as it's more stable and doesn't drift when results change mid-fetch. The select parameter further keeps responses efficient by only pulling the fields we actually need (id, publication_year, cited_by_count, authorships, title, abstract_inverted_index), reducing payload size per request.

Finally, since the same paper can appear for multiple authors in a batch, we deduplicate on id after collecting all results, so D2 and D3 contain only unique papers.

> - **Filtering Criteria and Dataset Relevance** Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?  

- Works count (5–5000): Lower bound removes researchers with almost no publication record. Upper bound drops extreme outliers like organization where the OpenAlex entry probably doesn't map to one real person anyway.

- Citation count (>10): Filters for work that had some actual impact. The obvious tradeoff is it biases against recent papers that haven't had time to accumulate citations, so newer CSS work is likely underrepresented.

- Author count (<10): Large author lists tend to signal big organization-style collaborations rather than the smaller teams typical in CSS. Cutting them keeps the network cleaner, but probably underrepresents fields like computational epidemiology that genuinely involve big teams.

- Field filters (Social Science AND Quantitative): Requiring both concept tags is a reasonable operationalisation of CSS, but OpenAlex's tagging isn't perfect. A network science paper tagged only under Physics gets dropped even if it's clearly CSS, so researchers coming from a harder quant background are likely underrepresented and vice versa.